In [1]:
import sys
import os
import glob
import librosa
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

In [2]:
def file_to_vector_array(file_name, n_mels=128, n_fft=1024, hop_length=512, power=2.0):
    y, sr = librosa.load(file_name, sr=None, mono=True)
    mel_spectrogram = librosa.feature.melspectrogram(y=y,
                                                     sr=sr,
                                                     n_fft=n_fft,
                                                     hop_length=hop_length,
                                                     n_mels=n_mels,
                                                     power=power)

    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max).astype(np.float32)
    vector_array = log_mel_spectrogram.T

    return vector_array

def files_to_sliding_windows_array(file_list, 
                                   n_mels=128, 
                                   frames=32, 
                                   n_fft=1024, 
                                   hop_length=512, 
                                   power=2.0,
                                   step=8):
    patches = []

    for file in file_list:
        vec = file_to_vector_array(file, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length, power=power)
        n_vectors = vec.shape[0]
        # Sliding window
        for start in range(0, n_vectors - frames + 1, step):
            patch = vec[start : start + frames, :n_mels]  # (frames, n_mels)
            if patch.shape[0] == frames:  # Seguridad
                patches.append(patch[..., np.newaxis])  # (frames, n_mels, 1)

    X = np.stack(patches)
    return X

In [3]:
def list_wavs_by_machine_id(train_dir, machine_id):
    training_list_path = os.path.relpath("../dataset/{dir}/*_id_{id}_*.wav".format(dir=train_dir, id=machine_id))
    files = sorted(glob.glob(training_list_path))
    return files

In [4]:
import keras.models
from keras.models import Model
from keras.layers import Input, BatchNormalization, Activation, Reshape, Flatten, Dense
from keras.layers import Conv2D, Cropping2D, Conv2DTranspose

def get_model(inputDim, latentDim):
    input_img = Input(shape=(inputDim[0], inputDim[1], 1))

    x = Conv2D(32, (5, 5),strides=(1,2), padding='same')(input_img)   #32x128 -> 32x64
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(64, (5, 5),strides=(1,2), padding='same')(x)           #32x32
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(128, (5, 5),strides=(2,2), padding='same')(x)          #16x16
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(256, (3, 3),strides=(2,2), padding='same')(x)          #8x8
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(512, (3, 3),strides=(2,2), padding='same')(x)          #4x4
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    volumeSize = x.shape
    # at this point the representation size is latentDim i.e. latentDim-dimensional
    x = Conv2D(latentDim, (4,4), strides=(1,1), padding='valid')(x)
    encoded = Flatten()(x)
    
    x = Dense(volumeSize[1] * volumeSize[2] * volumeSize[3])(encoded) 
    x = Reshape((volumeSize[1], volumeSize[2], 512))(x)                #4x4

    x = Conv2DTranspose(256, (3, 3),strides=(2,2), padding='same')(x)  #8x8
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2DTranspose(128, (3, 3),strides=(2,2), padding='same')(x)  #16x16   
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2DTranspose(64, (5, 5),strides=(2,2), padding='same')(x)   #32x32
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2DTranspose(32, (5, 5),strides=(1,2), padding='same')(x)   #32x64
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    decoded = Conv2DTranspose(1, (5, 5),strides=(1,2), padding='same')(x) 

    return Model(inputs=input_img, outputs=decoded)

In [5]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from keras.callbacks import EarlyStopping, ModelCheckpoint
import pandas as pd

machine_config = {
    'fan': ['00', '02', '04', '06'],
    'valve': ['00', '02', '04', '06'],
    'pump': ['00', '02', '04', '06'],
    'slider': ['00', '02', '04', '06'],
    'ToyCar': ['01', '02', '03', '04'],
    'ToyConveyor': ['01', '02', '03']
}

cnn_ae_results = []

for machine_type, machine_ids in machine_config.items():
    for m_id in machine_ids:
        print(f'--- (CNN AE) Processing {machine_type} with ID: {m_id} ---')
        train_files = list_wavs_by_machine_id(f'{machine_type}/train', m_id)

        train_files, val_files = train_test_split(train_files, test_size=0.1, random_state=42)

        train_data = files_to_sliding_windows_array(train_files)
        val_data = files_to_sliding_windows_array(val_files)
        
        train_data_mean = np.mean(train_data)
        train_data_std = np.std(train_data)

        train_data = (train_data - train_data_mean) / (train_data_std + 1e-7)
        val_data = (val_data - train_data_mean) / (train_data_std + 1e-7)

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True),
            ModelCheckpoint(filepath=f'../models/cnn_ae/cnn_ae_model_{machine_type}_{m_id}.keras', monitor='val_loss', mode='min', save_best_only=True)
        ]
        
        model = get_model((32, 128), 40)
        model.compile(optimizer='adam', loss='mse')
        
        model.fit(train_data, train_data, epochs=10, batch_size=64, validation_data=(val_data, val_data), callbacks=callbacks)
        
        test_files = list_wavs_by_machine_id(f'{machine_type}/test', m_id)
        y_true = [1 if 'anomaly' in os.path.basename(f) else 0 for f in test_files]

        y_pred = []

        for file in test_files:
            vec = file_to_vector_array(file)

            vec = (vec - train_data_mean) / (train_data_std + 1e-7)
            
            length, _ = vec.shape
            patches = []
            
            idex = np.arange(length - 32 + 1, step=8)

            for start in idex:
                patch = vec[start:start+32, :]
                patches.append(patch[..., np.newaxis])

            data = np.stack(patches)
            
            preds = model.predict(data, verbose=0)
            errors = np.mean(np.square(data - preds), axis=(1, 2, 3))
            y_pred.append(np.mean(errors))
        
        fpr, tpr, _ = roc_curve(y_true, y_pred)
        roc_auc = auc(fpr, tpr)

        cnn_ae_results.append({
            'Machine_Type': machine_type,
            'Machine_ID': m_id,
            'AUC_Score': roc_auc
        })
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC - {machine_type} with ID {m_id}')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        
        plt.savefig(f'../results/cnn_ae/{machine_type}_{m_id}_roc.png')
        plt.close()

--- (CNN AE) Processing fan with ID: 00 ---
Epoch 1/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 49s 73ms/step - loss: 0.1737 - val_loss: 0.2561
Epoch 2/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - loss: 0.1260 - val_loss: 0.1445
Epoch 3/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - loss: 0.1154 - val_loss: 0.1293
Epoch 4/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - loss: 0.1116 - val_loss: 0.1129
Epoch 5/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - loss: 0.1099 - val_loss: 0.1274
Epoch 6/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - loss: 0.1075 - val_loss: 0.1184
Epoch 7/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - loss: 0.1057 - val_loss: 0.1167
Epoch 8/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - loss: 0.1043 - val

In [6]:
results_df = pd.DataFrame(cnn_ae_results)
results_df

,Machine_Type,Machine_ID,AUC_Score
0,fan,00,0.524619
1,fan,02,0.713649
2,fan,04,0.571954
3,fan,06,0.836066
4,valve,00,0.659916
5,valve,02,0.733583
6,valve,04,0.754667
7,valve,06,0.597250
8,pump,00,0.680769
9,pump,02,0.561892
